# 3.2 注意力机制 (Attention Mechanism)

注意力机制是Transformer的核心，决定了模型如何捕获序列中的依赖关系。

本节涵盖：
- 多头注意力（MHA）
- 多查询注意力（MQA）
- 分组查询注意力（GQA）
- 多头潜在注意力（MLA）
- 稀疏注意力
- FlashAttention

## 1. 多头注意力（MHA）

**目的**：从多个子空间捕获不同类型的依赖关系

**基本原理**：将Q/K/V分别投影到多个头，每个头独立计算注意力后拼接，使模型能同时关注不同位置的不同表征子空间信息。

**KV Cache大小**：2 × n_heads × seq_len × d_head（每个token需要缓存K和V）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.Wq(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        K = self.Wk(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V = self.Wv(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.Wo(out), attn

mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
causal_mask = torch.tril(torch.ones(10, 10)).unsqueeze(0).unsqueeze(0)
out, attn_weights = mha(x, mask=causal_mask)

kv_cache_per_token = 2 * 4 * 16
print('=== Multi-Head Attention (MHA) ===')
print(f'Input: {x.shape}')
print(f'Output: {out.shape}')
print(f'Attention weights: {attn_weights.shape}')
print(f'KV cache per token: {kv_cache_per_token} floats (2 × {4} heads × {16} dim)')
print(f'Attention is causal: {attn_weights[0, 0].triu(1).sum().item() == 0}')

## 2. 多查询注意力（MQA）

**目的**：减少KV缓存，加速推理

**基本原理**：所有注意力头共享同一组K和V投影，仅Q保留多头，大幅减少KV Cache大小，提升推理速度。代表：PaLM、StarCoder。

**KV Cache节省**：从 2×n_heads×d_head 降至 2×1×d_head，节省 (n_heads-1)/n_heads

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class MultiQueryAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, self.d_head)
        self.Wv = nn.Linear(d_model, self.d_head)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.Wq(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        K = self.Wk(x).unsqueeze(1)
        V = self.Wv(x).unsqueeze(1)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.Wo(out), attn

mqa = MultiQueryAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
out, attn = mqa(x, mask=causal_mask)

mha_kv = 2 * 4 * 16
mqa_kv = 2 * 1 * 16
print('=== Multi-Query Attention (MQA) ===')
print(f'Output: {out.shape}')
print(f'\nKV cache per token comparison:')
print(f'  MHA: {mha_kv} floats (2 × 4 heads × 16 dim)')
print(f'  MQA: {mqa_kv} floats (2 × 1 head × 16 dim)')
print(f'  Savings: {(1 - mqa_kv/mha_kv)*100:.0f}% reduction in KV cache')
print(f'\nKey: All query heads share the same K and V, drastically reducing KV cache.')

## 3. 分组查询注意力（GQA）

**目的**：在MHA和MQA之间取得平衡

**基本原理**：将注意力头分组，每组共享一组K/V投影，既减少了KV Cache又保持了比MQA更好的模型质量。代表：LLaMA-2、Mistral。

**GQA是MHA和MQA的推广**：
- n_kv_heads = n_heads → MHA
- n_kv_heads = 1 → MQA
- 1 < n_kv_heads < n_heads → GQA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_kv_heads=2):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.Wv = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.Wq(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        K = self.Wk(x).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        V = self.Wv(x).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        K = K.unsqueeze(2).expand(-1, -1, self.n_rep, -1, -1).reshape(B, self.n_heads, S, self.d_head)
        V = V.unsqueeze(2).expand(-1, -1, self.n_rep, -1, -1).reshape(B, self.n_heads, S, self.d_head)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.Wo(out), attn

gqa = GroupedQueryAttention(d_model=64, n_heads=4, n_kv_heads=2)
x = torch.randn(2, 10, 64)
out, attn = gqa(x, mask=causal_mask)

mha_kv = 2 * 4 * 16
gqa_kv = 2 * 2 * 16
mqa_kv = 2 * 1 * 16
print('=== Grouped-Query Attention (GQA) ===')
print(f'Config: n_heads=4, n_kv_heads=2, d_head=16')
print(f'Output: {out.shape}')
print(f'\nKV cache per token comparison:')
print(f'  MHA (n_kv=4): {mha_kv} floats')
print(f'  GQA (n_kv=2): {gqa_kv} floats ({(1-gqa_kv/mha_kv)*100:.0f}% reduction)')
print(f'  MQA (n_kv=1): {mqa_kv} floats ({(1-mqa_kv/mha_kv)*100:.0f}% reduction)')
print(f'\nKey: GQA balances quality and efficiency between MHA and MQA.')

## 4. 多头潜在注意力（MLA）

**目的**：进一步压缩KV缓存

**基本原理**：使用低秩投影将KV压缩到低维潜在表示，推理时仅需缓存低维潜在向量，大幅降低内存占用。代表：DeepSeek-V2/V3。

**MLA的核心思想**：
- 不直接缓存高维的K和V
- 而是缓存低维的潜在向量c_kv
- 推理时从c_kv恢复K和V
- 缓存大小从 2×n_heads×d_head 降至 2×d_latent（d_latent << n_heads×d_head）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_latent=16):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_latent = d_latent
        self.Wq = nn.Linear(d_model, d_model)
        self.down_proj_k = nn.Linear(d_model, d_latent, bias=False)
        self.up_proj_k = nn.Linear(d_latent, d_model, bias=False)
        self.down_proj_v = nn.Linear(d_model, d_latent, bias=False)
        self.up_proj_v = nn.Linear(d_latent, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.Wq(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        c_kv = self.down_proj_k(x)
        K = self.up_proj_k(c_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        c_vv = self.down_proj_v(x)
        V = self.up_proj_v(c_vv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.Wo(out), c_kv, c_vv

mla = MultiHeadLatentAttention(d_model=64, n_heads=4, d_latent=16)
x = torch.randn(2, 10, 64)
out, c_kv, c_vv = mla(x, mask=causal_mask)

mha_kv = 2 * 4 * 16
mla_kv = 2 * 16
print('=== Multi-Head Latent Attention (MLA) ===')
print(f'Config: d_model=64, n_heads=4, d_latent=16')
print(f'Output: {out.shape}')
print(f'Latent KV: c_kv={c_kv.shape}, c_vv={c_vv.shape}')
print(f'\nKV cache per token comparison:')
print(f'  MHA: {mha_kv} floats (2 × 4 heads × 16 dim)')
print(f'  MLA: {mla_kv} floats (2 × 16 latent dim)')
print(f'  Savings: {(1 - mla_kv/mha_kv)*100:.0f}% reduction in KV cache')
print(f'\nKey: MLA caches low-dim latent vectors instead of full K/V,')
print(f'recovering K/V on-the-fly during attention computation.')

## 5. 稀疏注意力

**目的**：降低长序列的注意力计算复杂度

**基本原理**：仅让每个token关注部分关键token而非全部，将复杂度从O(n²)降至O(n)。常见模式：
- **局部窗口**：每个token只关注附近W个token
- **全局token**：少数token可以被所有token关注
- **随机注意力**：随机选择部分远距离token关注
- **组合**：局部+全局+随机（如Longformer、BigBird）

In [ ]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(42)

def create_sparse_mask(seq_len, window_size=3, n_global=2, n_random=1):
    mask = torch.zeros(seq_len, seq_len)
    for i in range(seq_len):
        left = max(0, i - window_size)
        right = min(seq_len, i + window_size + 1)
        mask[i, left:right] = 1
        mask[i, :n_global] = 1
        if i >= n_global:
            candidates = list(range(n_global, i - window_size)) + list(range(i + window_size + 1, seq_len))
            candidates = [c for c in candidates if c >= 0 and c < seq_len]
            if candidates:
                n_pick = min(n_random, len(candidates))
                picked = torch.tensor(candidates)[torch.randperm(len(candidates))[:n_pick]]
                mask[i, picked] = 1
    return mask

seq_len = 16
full_mask = torch.ones(seq_len, seq_len)
local_mask = torch.tril(torch.ones(seq_len, seq_len)) * torch.triu(torch.ones(seq_len, seq_len), diagonal=-3)
local_mask = (local_mask > 0).float()
sparse_mask = create_sparse_mask(seq_len, window_size=2, n_global=2, n_random=1)

full_sparsity = 1 - full_mask.sum() / full_mask.numel()
local_sparsity = 1 - local_mask.sum() / local_mask.numel()
sparse_sparsity = 1 - sparse_mask.sum() / sparse_mask.numel()

print('=== Sparse Attention Patterns ===')
print(f'Sequence length: {seq_len}\n')
print(f'Full attention sparsity: {full_sparsity:.1%} (0% = dense)')
print(f'Local window (w=3) sparsity: {local_sparsity:.1%}')
print(f'Local+Global+Random sparsity: {sparse_sparsity:.1%}')

print(f'\nSparse mask visualization (1=attend, 0=skip):')
for i in range(min(8, seq_len)):
    row = ''.join(['█' if sparse_mask[i,j] else '·' for j in range(min(16, seq_len))])
    print(f'  pos {i:2d}: {row}')

full_flops = seq_len ** 2
sparse_flops = int(sparse_mask.sum().item())
print(f'\nComputation: full={full_flops} ops, sparse={sparse_flops} ops ({(1-sparse_flops/full_flops)*100:.0f}% savings)')
print('\nKey: Sparse attention reduces O(n²) to ~O(n) by attending to only key positions.')

## 6. FlashAttention

**目的**：加速注意力计算并减少显存占用

**基本原理**：通过分块计算（tiling）和内核融合避免在GPU HBM中实例化完整的注意力矩阵，在数学等价的前提下大幅提升计算速度和显存效率。

**FlashAttention的关键优化**：
- **分块计算**：将Q/K/V分成小块，在SRAM中完成注意力计算
- **在线softmax**：使用增量式softmax避免存储完整注意力矩阵
- **内核融合**：将多个操作融合为一个GPU内核，减少HBM读写
- **PyTorch集成**：`F.scaled_dot_product_attention`已内置FlashAttention

In [ ]:
import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)

def manual_attention(Q, K, V, mask=None):
    d_head = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return attn @ V

B, n_heads, S, d_head = 4, 8, 256, 64
Q = torch.randn(B, n_heads, S, d_head, device='cpu')
K = torch.randn(B, n_heads, S, d_head, device='cpu')
V = torch.randn(B, n_heads, S, d_head, device='cpu')

out_manual = manual_attention(Q, K, V)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

diff = (out_manual - out_sdpa).abs().max().item()

print('=== FlashAttention via scaled_dot_product_attention ===')
print(f'Input: Q={Q.shape}, K={K.shape}, V={V.shape}')
print(f'Manual attention output: {out_manual.shape}')
print(f'SDPA output: {out_sdpa.shape}')
print(f'Max difference: {diff:.6f} (should be ~0, mathematically equivalent)')

sizes = [64, 128, 256, 512]
print(f'\nMemory comparison (theoretical):')
for S in sizes:
    manual_mem = B * n_heads * S * S * 4
    flash_mem = B * n_heads * S * d_head * 4 * 3
    print(f'  Seq={S:4d}: Manual={manual_mem/1e6:.1f}MB (attn matrix), Flash={flash_mem/1e6:.1f}MB (QKV only), savings={1-flash_mem/manual_mem:.1%}')

print(f'\nKey: FlashAttention avoids materializing the {S}×{S} attention matrix in HBM.')
print(f'This gives both speed improvement and memory savings, especially for long sequences.')